# Prepare IMC_NB_TumorSub dataset

Creates a new HDF5 dataset that keeps individual neuroblastoma tumour-cell subtypes
instead of merging them all into a single `Tumor` class.

**Key changes vs IMC_NB_FineCT:**
- `Tumor` split into 9 subtypes: `TC_early`, `TC_CXCR4hi`, `TC_bridge`, `TC_CD24neg`,
  `TC_CD24pos`, `TC_GD2lo`, `TC_GATA3hi`, `TC_CHGAhi`, `TC_CD44`
- **Bug fix:** `CD44+ TC` was incorrectly mapped to `T_Cell` in IMC_NB_FineCT — corrected to `TC_CD44`
- All other classes (immune, stromal, progenitor) identical to IMC_NB_FineCT
- `Other` present but set as `ignore_annotation` in dataset config

Source: `annotations.csv` from `IMC_Neuroblastoma/` (original fine-grained labels)

Output: `h5_files/IMC_NB_TumorSub/IMC_NB_TumorSub.h5` + splits + `used_markers.txt`

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import h5py
import tifffile
from natsort import natsorted
from collections import Counter, defaultdict
from tqdm import tqdm

dataset_base  = Path('/home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/data/MCI_data')
source_name   = 'IMC_Neuroblastoma'
new_name      = 'IMC_NB_TumorSub'

source_path      = dataset_base / source_name
output_h5_folder = dataset_base / f'h5_files/{new_name}'
output_h5_folder.mkdir(parents=True, exist_ok=True)
output_h5_path   = output_h5_folder / f'{new_name}.h5'

print(f'Source:  {source_path}')
print(f'Output:  {output_h5_path}')

## 1 · Load source annotations and show original distribution

In [ ]:
annotations  = pd.read_csv(source_path / 'annotations.csv')
marker_names = pd.read_csv(source_path / 'marker_names.csv', index_col=0)['Label'].to_list()

print(f'Total cells: {len(annotations):,}')
print(f'Markers:     {len(marker_names)}')
print()
print('Original fine-grained distribution:')
vc = annotations['annotation'].value_counts()
for ct, n in vc.items():
    print(f'  {ct:>35}  {n:>7,}  ({100*n/len(annotations):5.2f}%)')

## 2 · Define new merging map

Key decisions:
- Tumour subtypes kept individually (drop GATA3hi / CHGAhi later if too small)
- **`CD44+ TC` → `TC_CD44`** (bug fix: was `T_Cell` in IMC_NB_FineCT)
- Immune / stromal / progenitor classes identical to IMC_NB_FineCT

In [ ]:
tumor_sub_map = {
    # ── Tumour subtypes (kept separate) ───────────────────────────────────
    'early NB TC':        'TC_early',    # 49,538 — large classical NB state
    'CXCR4hi TC':         'TC_CXCR4hi', # 31,911 — migratory/aggressive state
    'bridge-like TC':     'TC_bridge',  # 24,200 — transitional state
    'CD24- marker-lo TC': 'TC_CD24neg', # 19,081 — low-marker quiescent
    'CD24+ marker-lo TC': 'TC_CD24pos', # 18,365 — low-marker quiescent (CD24+)
    'CD44+ TC':           'TC_CD44',    # 33,873 — BUG FIX (was T_Cell in FineCT)
    'GD2lo TC':           'TC_GD2lo',  #  5,322 — GD2-low tumour cells
    'GATA3hi TC':         'TC_GATA3hi',#  1,430 — keep for now, can drop later
    'CHGAhi TC':          'TC_CHGAhi', #    684 — keep for now, can drop later

    # ── Proliferating (Ki-67+, kept merged) ───────────────────────────────
    'Ki67hi TC':          'Proliferating',  # 31,051
    'LIN- Ki67+':         'Proliferating',  #    789

    # ── Stromal ───────────────────────────────────────────────────────────
    'fibroblast/endothel': 'Stromal',  # 36,418
    'schwann cell':        'Stromal',  #  2,036

    # ── T cells — CD4 / CD8 / unknown ─────────────────────────────────────
    'CD4+ naive T cell':       'CD4_T',
    'CD4+ PD1+ T cell':        'CD4_T',
    'CD8+ naive T cell':       'CD8_T',
    'CD8+ PDL1lo T cell':      'CD8_T',
    'CD8+ GZMB+ PD1lo T cell': 'CD8_T',
    'CD8+ S100B+ T cell':      'CD8_T',
    'CD3+ T cell':              'T_Cell',
    'CD3+ GZMB+ T cell':        'T_Cell',
    'dense T cell region':      'T_Cell',

    # ── NK / DC ───────────────────────────────────────────────────────────
    'GZMB+ S100B- DC/NK': 'NK_DC',  # 111

    # ── Myeloid ───────────────────────────────────────────────────────────
    'neutrophil':        'Neutrophil',
    'monoblast-like':    'Myeloid',
    'CD14- PDL1+ MO/DC': 'Myeloid',
    'CD14+ PDL1+ MO':    'Myeloid',
    'CD14+ PDL1- MO':    'Myeloid',
    'pDC':               'Myeloid',

    # ── B cells ───────────────────────────────────────────────────────────
    'B cell': 'B_Cell',

    # ── Progenitors ───────────────────────────────────────────────────────
    'HPC':               'Progenitor',
    'neural progenitor': 'Progenitor',
    'Ki67+ CXCR4+ cell': 'Progenitor',

    # ── Other (ignored during training) ───────────────────────────────────
    'other': 'Other',
}

print(f'Mapping covers {len(tumor_sub_map)} original types → {len(set(tumor_sub_map.values()))} merged classes:')
reverse = defaultdict(list)
for k, v in tumor_sub_map.items():
    reverse[v].append(k)
for cls in sorted(reverse):
    print(f'  {cls:<15}  ←  {", ".join(reverse[cls])}')

## 3 · Apply mapping and verify coverage

In [ ]:
annotations['annotation_new'] = annotations['annotation'].map(tumor_sub_map)

unmapped = annotations.loc[annotations['annotation_new'].isna(), 'annotation'].unique()
if len(unmapped) > 0:
    print(f'WARNING — unmapped types (will be dropped): {unmapped}')
else:
    print('All cell types mapped successfully.')

ann_filtered = annotations.dropna(subset=['annotation_new']).copy()
print(f'\nCells before: {len(annotations):,}  →  after: {len(ann_filtered):,}  (dropped {len(annotations)-len(ann_filtered):,})')

print('\nNew distribution (all classes including Other):')
vc_new = ann_filtered['annotation_new'].value_counts()
for ct, n in vc_new.items():
    print(f'  {ct:<15}  {n:>7,}  ({100*n/len(ann_filtered):5.2f}%)')

## 4 · Create HDF5

In [ ]:
sample_ids = natsorted(ann_filtered['SampleID'].unique().tolist())
print(f'Samples: {len(sample_ids)}')

annotation_strings = ann_filtered['annotation_new'].astype(str).tolist()

with h5py.File(output_h5_path, 'w') as h5f:
    dt = h5py.special_dtype(vlen=str)

    marker_ds = h5f.create_dataset('marker_names', (len(marker_names),), dtype=dt)
    marker_ds[:] = marker_names

    coords = h5f.create_group('coords')
    coords.create_dataset('DIM1',      data=ann_filtered['DIM1'].values)
    coords.create_dataset('DIM2',      data=ann_filtered['DIM2'].values)
    coords.create_dataset('sample_id', data=ann_filtered['SampleID'].astype(str).tolist(), dtype=dt)

    h5f.create_dataset('annotation', data=annotation_strings, dtype=dt)
    h5f.create_dataset('sample_ids', data=sample_ids, dtype=dt)

    data_grp = h5f.create_group('data')
    for sid in tqdm(sample_ids, desc='Writing samples'):
        img   = tifffile.imread(source_path / f'images/{sid}_image.tif')
        masks = tifffile.imread(source_path / f'masks/{sid}_masks.tif').astype(np.uint32)
        sg = data_grp.create_group(sid)
        sg.create_dataset('image',  data=img,   compression='gzip', compression_opts=3, chunks=(64, 64, 1))
        sg.create_dataset('masks',  data=masks, compression='gzip', compression_opts=3, chunks=(64, 64))

print(f'\nSaved → {output_h5_path}')

## 5 · Verify H5 structure

In [ ]:
with h5py.File(output_h5_path, 'r') as f:
    print('Root keys:    ', list(f.keys()))
    print('Samples:      ', len(f['sample_ids']))
    print('Annotations:  ', len(f['annotation']))
    print('Unique classes:', sorted(np.unique(f['annotation'][()].astype(str))))
    sid0 = f['sample_ids'][0].decode()
    print(f'Example [{sid0}] image shape:', f['data'][sid0]['image'].shape)
    print(f'Example [{sid0}] masks shape:', f['data'][sid0]['masks'].shape)

## 6 · Generate train / val / test splits (70 / 10 / 20, stratified per class)

In [ ]:
annotations_arr = np.array(annotation_strings)
counts          = Counter(annotations_arr)
total           = len(annotations_arr)

splits = dict(train=0.7, val=0.1, test=0.2)
assert sum(splits.values()) == 1.0

train_idxs, val_idxs, test_idxs = [], [], []
rng = np.random.default_rng(42)

for celltype, n in counts.items():
    idxs = np.where(annotations_arr == celltype)[0]
    rng.shuffle(idxs)
    t = int(n * splits['train'])
    v = int(n * splits['val'])
    train_idxs.extend(sorted(idxs[:t]))
    val_idxs.extend(sorted(idxs[t:t+v]))
    test_idxs.extend(sorted(idxs[t+v:]))

assert len(train_idxs) + len(val_idxs) + len(test_idxs) == total

np.savetxt(output_h5_folder / 'train.txt', sorted(train_idxs), fmt='%d')
np.savetxt(output_h5_folder / 'val.txt',   sorted(val_idxs),   fmt='%d')
np.savetxt(output_h5_folder / 'test.txt',  sorted(test_idxs),  fmt='%d')

print(f'Train: {len(train_idxs):,}  |  Val: {len(val_idxs):,}  |  Test: {len(test_idxs):,}')
print('Saved train.txt, val.txt, test.txt')

## 7 · Create used_markers.txt

Same exclusion list as `IMC_NeuroblastomaMetaCluster` and `IMC_NB_FineCT`.

In [ ]:
exclude = {
    'IF1', 'IF2', 'IF3',       # internal fiducials
    'MPO',                      # low SNR in IMC
    'H3K9Ac', 'H4K12Ac',       # histone marks — noisy
    'CXCR2',                    # low expression
    'IDO',                      # low expression
    'clPARP',                   # apoptosis — not cell-type specific
    'PNMT',                     # low SNR
    'DNA1',                     # structural, not protein
    'Fibronectin',              # ECM — not cell-type specific
    'FOXP3',                    # low SNR in IMC (nuclear TF)
}

with h5py.File(output_h5_path, 'r') as f:
    all_markers = set(f['marker_names'][()].astype(str))

kept = sorted(all_markers - exclude)
print(f'Total markers: {len(all_markers)}  →  kept: {len(kept)}  (excluded: {len(exclude)})')
print('Kept:', kept)

np.savetxt(output_h5_folder / 'used_markers.txt', kept, fmt='%s')
print('\nSaved used_markers.txt')

## 8 · Summary — dataset config values

In [ ]:
n_markers = len(kept)
n_classes = len(set(tumor_sub_map.values()) - {'Other'})

print('=' * 60)
print(f'  Dataset:    {new_name}')
print(f'  H5 path:    {output_h5_path}')
print(f'  n_markers:  {n_markers}   (same as IMC_NB_FineCT)')
print(f'  n_classes:  {n_classes}   (excl. Other)')
print(f'  patch_size: 24   (same)')
print(f'  cutter_size:12   (same)')
print(f'  ignore_annotation: ["Other", "NK_DC"]   # NK_DC has only 111 cells')
print('=' * 60)
print()
print('Cell type counts (excl. Other):')
for ct, n in vc_new.items():
    if ct == 'Other':
        continue
    flag = '  ← small' if n < 500 else ('  ← can drop' if n < 1000 else '')
    print(f'  {ct:<15}  {n:>7,}{flag}')